In [ ]:
# libraries
import os
import json
import time
import subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import nbformat
from IPython.display import display

In [ ]:
# globals; run through run_pipeline.ipynb
ROOT = Path(os.environ['NCA_PROJECT_ROOT'])
TASK = ROOT / 'g_phase2_test'
INPUT, OUTPUT = TASK / 'input', TASK / 'output'
CFG = json.loads((INPUT / 'config.json').read_text())
DEVICE = torch.device(CFG['device'])
torch.set_num_threads(CFG['cpu_threads'])
torch.use_deterministic_algorithms(True)
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False

In [ ]:
# functions passed through c_models/output; definitions only, no training side effects
runtime_nb = nbformat.read(INPUT / 'runtime.ipynb', as_version=4)
for runtime_cell in runtime_nb.cells:
    exec(compile(runtime_cell.source, 'runtime.ipynb', 'exec'), globals())
PACK = torch.load(INPUT / 'dataset.pt', weights_only=True, map_location='cpu')
SPECS = CFG['models']

In [ ]:
# trained run imports
manifest = pd.read_csv(INPUT / 'phase2_manifest.csv')
metric_frames, sample_frames, channel_rows, transition_rows, transition_summaries = [], [], [], [], []
dynamics_frames, examples, animations = [], [], []

In [ ]:
# Phase 2 held-out reconstruction; wrong/ambiguous outputs cannot count as class generation
with torch.no_grad():
    for run in manifest.to_dict('records'):
        spec = next(item for item in SPECS if item['name'] == run['model'])
        model, _ = load_trained(INPUT / 'phase2_runs' / run['checkpoint'], spec, CFG, DEVICE, run['replicate'])
        generator = torch_stream(CFG, DEVICE, 80 + run['replicate'])
        rng = numpy_stream(CFG, 4000 + run['replicate'])
        metadata = {key: run[key] for key in ['model', 'regime', 'replicate']}
        for split in ['test_composition', 'test_angle', 'test_color', 'test_translation', 'test_joint']:
            ids = panel_ids(PACK, split, CFG)
            probabilities = np.resize([0.1, 0.4, 0.6, 0.9], len(ids))
            latents = rng.uniform(size=len(ids))
            target, labels = select_target(PACK, ids, probabilities, latents, DEVICE)
            prototypes = PACK['targets'][ids].float().to(DEVICE)
            for rate in CFG['stress_flip_rates']:
                for trial in range(CFG['channel_trials']):
                    state = generate_panel(model, PACK, CFG, ids, probabilities, latents, run['regime'], generator, flips=all_flips(rate))
                    predicted, valid, quality, margin = classify(state, prototypes, CFG)
                    frame = metric_rows(state, target, {**metadata, 'condition': ids, 'split': split, 'trial': trial,
                                        'flip_rate': rate, 'intervention': 'none', 'probability': probabilities,
                                        'latent_u': latents, 'intended_class': labels.cpu().numpy()})
                    frame['predicted_class'] = predicted.cpu().numpy()
                    frame['valid'] = valid.cpu().numpy()
                    frame['successful_generation'] = (valid & predicted.eq(labels)).cpu().numpy()
                    metric_frames.append(frame)
                    if rate == 0 and trial == 0:
                        cue = make_cue(PACK, ids[:1], probabilities[:1], latents[:1], CFG, DEVICE)
                        channel_rows.extend(channel_diagnostics(model, state[:1], cue, generator, metadata))
                        for index in range(min(2, len(ids))):
                            examples.append({**metadata, 'split': split, 'condition': int(ids[index]),
                                             'class': int(labels[index]), 'target': target[index].cpu(),
                                             'generated': state[index, ..., :4].cpu()})
        # cat-only recall uses the same seed controls as Phase 1
        ids = panel_ids(PACK, 'test_composition', CFG)
        p, u = np.zeros(len(ids)), np.full(len(ids), 0.5)
        target, _ = select_target(PACK, ids, p, u, DEVICE)
        state = generate_panel(model, PACK, CFG, ids, p, u, run['regime'], generator)
        metric_frames.append(metric_rows(state, target, {**metadata, 'condition': ids, 'split': 'test_composition',
                             'trial': 0, 'flip_rate': 0.0, 'intervention': 'cat_retention'}))
        display(pd.DataFrame([{**metadata, 'test': 'conditional generation and noise sweep', 'status': 'complete'}]))

In [ ]:
# mixture calibration; reuse latent draws across p values, resample communication for each trial
# independent unit is a latent draw, not each repeated channel realization
with torch.no_grad():
    for run in manifest.to_dict('records'):
        spec = next(item for item in SPECS if item['name'] == run['model'])
        model, _ = load_trained(INPUT / 'phase2_runs' / run['checkpoint'], spec, CFG, DEVICE, run['replicate'])
        generator = torch_stream(CFG, DEVICE, 90 + run['replicate'])
        rng = numpy_stream(CFG, 4100 + run['replicate'])
        metadata = {key: run[key] for key in ['model', 'regime', 'replicate']}
        count = CFG['latent_samples']
        ids = np.resize(panel_ids(PACK, 'test_composition', CFG), count)
        latents = rng.uniform(size=count)
        prototypes = PACK['targets'][ids].float().to(DEVICE)
        for probability in CFG['calibration_ps']:
            probabilities = np.full(count, probability)
            target, labels = select_target(PACK, ids, probabilities, latents, DEVICE)
            for trial in range(CFG['channel_trials']):
                state = generate_panel(model, PACK, CFG, ids, probabilities, latents, run['regime'], generator)
                predicted, valid, quality, margin = classify(state, prototypes, CFG)
                sample_frames.append(pd.DataFrame({**metadata, 'probability': probability, 'latent_id': np.arange(count),
                                     'condition': ids, 'latent_u': latents, 'trial': trial, 'intended_class': labels.cpu().numpy(),
                                     'predicted_class': predicted.cpu().numpy(), 'valid': valid.cpu().numpy(),
                                     'successful_generation': (valid & predicted.eq(labels)).cpu().numpy(),
                                     'quality_ratio': quality.cpu().numpy(), 'class_margin': margin.cpu().numpy(),
                                     'mse': image_metrics(state, target)['mse'].cpu().numpy()}))

In [ ]:
# online cue switches; classify the initial grown state before claiming adaptation
# assess latency only after two successive successful probes; failure is right-censored
switches = [(0.0, 1.0), (1.0, 0.0)] if CFG['profile'] == 'smoke' else [(0.0, 1.0), (1.0, 0.0), (0.2, 0.8), (0.8, 0.2)]
with torch.no_grad():
    for run in manifest.to_dict('records'):
        spec = next(item for item in SPECS if item['name'] == run['model'])
        model, _ = load_trained(INPUT / 'phase2_runs' / run['checkpoint'], spec, CFG, DEVICE, run['replicate'])
        generator = torch_stream(CFG, DEVICE, 100 + run['replicate'])
        metadata = {key: run[key] for key in ['model', 'regime', 'replicate']}
        ids = panel_ids(PACK, 'test_composition', CFG)[:CFG['batch_size']]
        latents = np.full(len(ids), 0.5)
        prototypes = PACK['targets'][ids].float().to(DEVICE)
        steps = steps_for(model, run['regime'], CFG['eval_steps'])
        probes = sorted(set([0, max(1, steps // 4), max(1, steps // 2), steps, steps + max(1, steps // 2), 2 * steps]))
        for old_p, new_p in switches:
            old_probabilities, new_probabilities = np.full(len(ids), old_p), np.full(len(ids), new_p)
            old_cue = make_cue(PACK, ids, old_probabilities, latents, CFG, DEVICE)
            new_cue = make_cue(PACK, ids, new_probabilities, latents, CFG, DEVICE)
            state, _ = run_steps(model, seed_state(PACK, ids, CFG, DEVICE), old_cue, steps, generator)
            before_class, before_valid, _, _ = classify(state, prototypes, CFG)
            intended_before = int(0.5 < old_p)
            intended_after = int(0.5 < new_p)
            eligible = (before_valid & before_class.eq(intended_before)).cpu().numpy()
            target, _ = select_target(PACK, ids, new_probabilities, latents, DEVICE)
            frames = {0: state.cpu()}
            state, later_frames = run_steps(model, state, new_cue, 2 * steps, generator, record=probes[1:])
            frames.update(later_frames)
            success_history = []
            for probe in probes:
                frame = frames[probe].to(DEVICE)
                predicted, valid, _, _ = classify(frame, prototypes, CFG)
                success = (valid & predicted.eq(intended_after)).cpu().numpy()
                success_history.append(success)
                metrics = image_metrics(frame, target)
                for index, condition in enumerate(ids):
                    transition_rows.append({**metadata, 'condition': int(condition), 'old_p': old_p, 'new_p': new_p,
                                            'steps_after_switch': probe, 'bits_after_switch': bit_cost(model, probe),
                                            'eligible_start': bool(eligible[index]), 'successful_response': bool(success[index]),
                                            'mse': float(metrics['mse'][index]), 'foreground_rmse': float(metrics['foreground_rmse'][index])})
            successes = np.stack(success_history)
            for index, condition in enumerate(ids):
                stable_at = None
                for j in range(CFG['stable_success_points'] - 1, len(probes)):
                    if successes[j - CFG['stable_success_points'] + 1:j + 1, index].all():
                        stable_at = probes[j]
                        break
                transition_summaries.append({**metadata, 'condition': int(condition), 'old_p': old_p, 'new_p': new_p,
                                             'eligible_start': bool(eligible[index]), 'responded_by_horizon': stable_at is not None,
                                             'adapted_from_valid_start': bool(eligible[index]) and stable_at is not None,
                                             'latency_steps': stable_at, 'censoring_horizon': 2 * steps,
                                             'latency_bits': None if stable_at is None else bit_cost(model, stable_at)})
            if run['replicate'] == 0 and run['regime'] == 'updates' and old_p == 0 and new_p == 1:
                animations.append({**metadata, 'probes': probes, 'frames': torch.stack([frames[t][0, ..., :4] for t in probes]),
                                   'target': target[0].cpu(), 'eligible_start': bool(eligible[0])})

In [ ]:
# persistence, self-repair and explicit channel dependence for Phase 2
with torch.no_grad():
    for run in manifest.to_dict('records'):
        spec = next(item for item in SPECS if item['name'] == run['model'])
        model, _ = load_trained(INPUT / 'phase2_runs' / run['checkpoint'], spec, CFG, DEVICE, run['replicate'])
        generator = torch_stream(CFG, DEVICE, 110 + run['replicate'])
        metadata = {key: run[key] for key in ['model', 'regime', 'replicate']}
        ids = panel_ids(PACK, 'test_composition', CFG)[:CFG['batch_size']]
        p, u = np.resize([0.0, 1.0], len(ids)), np.full(len(ids), 0.5)
        cue = make_cue(PACK, ids, p, u, CFG, DEVICE)
        target, labels = select_target(PACK, ids, p, u, DEVICE)
        prototypes = PACK['targets'][ids].float().to(DEVICE)
        steps = steps_for(model, run['regime'], CFG['eval_steps'])
        grown = generate_panel(model, PACK, CFG, ids, p, u, run['regime'], generator)
        damaged = centered_damage(grown, PACK['coordinates'][ids], CFG)
        repaired, _ = run_steps(model, damaged, cue, steps, generator)
        persistent, _ = run_steps(model, grown, cue, steps, generator)
        for name, state in [('initial', grown), ('damage_initial', damaged), ('repair', repaired), ('persistence', persistent)]:
            predicted, valid, _, _ = classify(state, prototypes, CFG)
            frame = metric_rows(state, target, {**metadata, 'condition': ids, 'intervention': name})
            frame['successful_generation'] = (valid & predicted.eq(labels)).cpu().numpy()
            dynamics_frames.append(frame)
        channels = [('m',)] if spec['family'] == 'message' else [('q',), ('r',), ('q', 'r')]
        for muted in channels:
            state = generate_panel(model, PACK, CFG, ids, p, u, run['regime'], generator, mute=muted)
            predicted, valid, _, _ = classify(state, prototypes, CFG)
            frame = metric_rows(state, target, {**metadata, 'condition': ids, 'split': 'test_composition', 'trial': 0,
                                'flip_rate': 0.0, 'intervention': 'mute_' + ''.join(muted)})
            frame['successful_generation'] = (valid & predicted.eq(labels)).cpu().numpy()
            metric_frames.append(frame)

In [ ]:
# exports; preserve every latent/channel realization so inference can respect its sampling unit
pd.concat(metric_frames, ignore_index=True).to_csv(OUTPUT / 'phase2_metrics.csv', index=False)
pd.concat(sample_frames, ignore_index=True).to_csv(OUTPUT / 'phase2_calibration_samples.csv', index=False)
pd.concat(dynamics_frames, ignore_index=True).to_csv(OUTPUT / 'phase2_dynamics.csv', index=False)
pd.DataFrame(channel_rows).to_csv(OUTPUT / 'phase2_channels.csv', index=False)
pd.DataFrame(transition_rows).to_csv(OUTPUT / 'phase2_transitions.csv', index=False)
pd.DataFrame(transition_summaries).to_csv(OUTPUT / 'phase2_transition_summary.csv', index=False)
torch.save(examples, OUTPUT / 'phase2_examples.pt')
torch.save(animations, OUTPUT / 'phase2_animations.pt')
display(pd.concat(metric_frames).query("intervention == 'none' and flip_rate == 0").groupby(['model', 'regime', 'split'])[
        ['mse', 'foreground_rmse', 'alpha_iou', 'successful_generation']].mean().reset_index())